In [1]:
import pandas as pd


In [2]:
import pandas as pd

df = pd.read_csv('/home/san/Archive/Computer_Engineering/Daneshkar_AI_Bootcamp/level_05_NLP_LLM_GenAI/homework/answer/dataset/TMDB_500_movie_dataset/tmdb_5000_movies.csv')

df = df[['title', 'overview']]

df.dropna(subset=['overview'], inplace=True)

df = df[df['overview'].str.strip() != '']

df.reset_index(drop=True, inplace=True)

print(f"تعداد فیلم‌های باقی‌مانده: {df.shape[0]}")

print(df.head())

تعداد فیلم‌های باقی‌مانده: 4799
                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   
3                     The Dark Knight Rises   
4                               John Carter   

                                            overview  
0  In the 22nd century, a paraplegic Marine is di...  
1  Captain Barbossa, long believed to be dead, ha...  
2  A cryptic message from Bond’s past sends him o...  
3  Following the death of District Attorney Harve...  
4  John Carter is a war-weary, former military ca...  


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(df['overview'])

print(f"(movie numbers,word number)")
print(tfidf_matrix.shape)

(movie numbers,word number)
(4799, 20978)


In [4]:
feature_names = tfidf.get_feature_names_out()

print(len(feature_names))
print(feature_names[:20])

20978
['00' '000' '007' '07am' '10' '100' '1000' '101' '108' '10th' '11' '114'
 '117' '118' '119' '11th' '12' '1200' '1215' '1250']


In [5]:
from sklearn.metrics.pairwise import cosine_similarity

def get_movie_recommendation(movie_title):


    titles = df['title'].tolist()

    if movie_title not in titles:
        return f"فیلمی با نام '{movie_title}' در دیتاست یافت نشد."

    movie_idx = titles.index(movie_title)

    similarity_scores = cosine_similarity(
        tfidf_matrix[movie_idx],
        tfidf_matrix
    ).flatten()

    ranked_indices = similarity_scores.argsort()[::-1]

    top_5_movies = []

    for idx in ranked_indices:
        if idx == movie_idx:
            continue

        top_5_movies.append({
            'title': df['title'].iloc[idx],
            'similarity_score': round(float(similarity_scores[idx]), 4)
        })

        if len(top_5_movies) == 5:
            break

    return pd.DataFrame(top_5_movies)

In [6]:
recommendations = get_movie_recommendation("The Dark Knight")
print(recommendations)

                                     title  similarity_score
0                    The Dark Knight Rises            0.3015
1                           Batman Returns            0.2494
2  Batman: The Dark Knight Returns, Part 2            0.2245
3                           Batman Forever            0.2141
4                                   Batman            0.1823


تحلیل نتایج:

۱. تشخیص خودکار مجموعه فیلم‌ها: چون در ستون خلاصه داستان همه فیلم‌های مرتبط با مرد خفاشی، واژه‌های مشترکی مانند مرد خفاشی، شهر خیالی، شخصیت منفی، قهرمان، مجرمان و شهر وجود دارد، مدل وزن‌دهی واژگان به این واژه‌ها امتیاز بالایی داده و شباهت کسینوسی بسیار زیادی بین بردارهای آن‌ها محاسبه کرده است.

۲. درباره فیلم‌های غیرمنتظره: ممکن است بپرسید چرا فیلم‌هایی مانند فیلم سیاسی مربوط به ترور یا یک فیلم جنایی آرام در بین این فهرست قرار گرفته‌اند. دلیل آن این است که این موتور فقط بر اساس واژه‌های خلاصه داستان کار می‌کند و مفهوم گونه سینمایی را نمی‌شناسد. خلاصه داستان فیلم سیاسی مربوط به ترور احتمالاً شامل واژه‌هایی مانند بررسی، جرم، قتل، توطئه و پلیس است. این واژه‌ها با خلاصه داستان شوالیه تاریکی، که درباره مبارزه با جرم و جنایت سازمان‌یافته است، هم‌پوشانی واژگانی دارند؛ بنابراین امتیاز شباهت گرفته‌اند.